# SaemPhysics: render a reviewed lesson draft in Google Colab

Codespaces/GitHub is the source of truth for code and small, reviewed metadata. This notebook clones that source, builds its local SQLite index, compiles the original kinematics integration example, and renders a Manim preview. Generated videos stay in Colab until you download them. This does **not** approve the physics or permit publication of source-paper content.

In [ ]:
from pathlib import Path
import subprocess

REPOSITORY = "https://github.com/saemsir/physics.git"
REVISION = "main"  # Prefer a reviewed commit SHA for repeatable production renders.
ROOT = Path("/content/physics")
if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)], check=True)
else:
    print(f"Using existing checkout: {ROOT}")
subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", "FETCH_HEAD"], check=True)
%cd /content/physics
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

## Install the renderer

This can take several minutes on a fresh Colab runtime. The Python requirements are pinned in the repository; system packages provide Cairo, Pango, LaTeX, and FFmpeg.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libcairo2-dev libpango1.0-dev texlive-latex-extra texlive-fonts-recommended dvisvgm
!python -m pip install -q -r requirements.txt

## Build the linked database and compile a lesson draft

The demo is an original engine test case, not a question from the IIT-JEE archive. Replace it only with a question whose source, rights, wording, concept tags, assumptions, and answer have been reviewed.

In [ ]:
!python -m content_engine init
!python -m content_engine summary
!python -m content_engine compile-lesson --problem data/kinematics_demo.json

## Render and inspect

This is a low-quality preview for review, not a release export. Check the model assumptions, numerical result, axes, labels, timing, and legibility before approval.

In [ ]:
!python -m manim -ql episodes/kinematics_001/scenes.py KinematicsMotion --media_dir /content/physics-render
from IPython.display import Video, display
renders = list(Path("/content/physics-render").rglob("KinematicsMotion.mp4"))
if not renders:
    raise FileNotFoundError("Manim completed without producing KinematicsMotion.mp4")
display(Video(str(renders[0]), embed=True))

## Download a preview copy (optional)

Keep the reviewed source, lesson IR, and approval decisions in Git. Keep large generated renders outside normal Git history.

In [ ]:
from google.colab import files
files.download(str(renders[0]))